# Kaggle runs for A2, A3 and C2

**Kaggle settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **On**. For the Hugging Face upload, add the secret `HF_TOKEN` (Add-ons > Secrets) and turn it on for this notebook.

**How to run a step:** set `STEP` in the next cell, then **Save Version > Save & Run All (Commit)**. You can close the browser. When it is done, open the version, go to **Output**, and download `to_commit.zip` (and `logits_*.zip` for Elif).

| STEP | When | What it does | About |
|---|---|---|---|
| `smoke` | any time | Runs all the code on a small fake dataset on the T4 (nothing to commit) | 30 min |
| `train_night` | after Gate 1 (clean data in the team repo) | C2 XLS-R on GPU 0 and A2 CTC on GPU 1 at the same time, then dev predictions and C2 dev logits | under 11.5 h |
| `train_day` | after `train_night` | A3 RNN-T, then its dev predictions | under 11.5 h |
| `test` | Saturday, after the freeze | USC test for A2, A3 and C2, C2 test logits | about 1 h |
| `upload` | any time after `train_night` | Uploads the C2 model to Hugging Face again (model card refresh) | 10 min |

For `test` and `upload`, first attach the outputs of the `train_night` and `train_day` versions: **Add Input > Your Work > Notebooks > this notebook**, choose the version. The notebook finds the checkpoints by itself.

Test sets are used once: run `test` only after the freeze (plan B4d).

In [ ]:
# ---- the only lines you change ----
STEP = "smoke"          # smoke | train_night | train_day | test | upload
REPO_URL = "https://github.com/ata-sesli/nlp-project-3"         # team repo: the data comes from here
DATA_BRANCH = "main"    # branch of the team repo that has Ata's data/ folder
CODE_REPO_URL = "https://github.com/Enes-Ozbek/nlp-project-3"   # your own repo: this code and your results
CODE_BRANCH = "enes-models"
HF_REPO = "Waxech/xls-r-300m-uzbek-usc"   # C2 model on Hugging Face ("" = no upload)
HOURS = 10.0            # training time per model; Kaggle stops a run at 12 h

In [ ]:
import glob, os, shutil, subprocess, time, zipfile
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content/work")
WORK.mkdir(parents=True, exist_ok=True)
REPO = Path("/tmp/repo")   # not in /kaggle/working, so the 2 GB of audio is not saved as output
SYSTEMS = ["A2_conformer-ctc", "A3_conformer-rnnt", "C2_xlsr-300m-ctc"]


def run(cmd, env=None):
    print(f"\n$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=REPO if REPO.exists() else None, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Command failed: {cmd}")


def run_parallel(jobs):
    """jobs = [(name, command)]; job k runs on GPU k. With one GPU they run one after the other."""
    import torch
    if torch.cuda.device_count() < len(jobs):
        print("Fewer GPUs than jobs: running one after the other (choose 'GPU T4 x2' to run them together).")
        for _, cmd in jobs:
            run(cmd)
        return
    procs = []
    for k, (name, cmd) in enumerate(jobs):
        log = WORK / f"log_{name}.txt"
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k))
        print(f"Starting {name} on GPU {k}, log: {log}")
        procs.append((name, log, subprocess.Popen(f"{cmd} > {log} 2>&1", shell=True, cwd=REPO, env=env)))
    while any(p.poll() is None for _, _, p in procs):
        time.sleep(600)
        for name, log, _ in procs:
            tail = log.read_text(errors="ignore").strip().splitlines()[-2:]
            print(f"[{time.strftime('%H:%M')}] {name}: {' | '.join(tail)[-300:]}", flush=True)
    for name, log, p in procs:
        print(f"\n===== {name} (exit code {p.returncode}) =====\n" + "\n".join(log.read_text(errors="ignore").splitlines()[-15:]))
    bad = [name for name, _, p in procs if p.returncode != 0]
    if bad:
        raise RuntimeError(f"Failed: {bad}. See the log files in the output.")


def pip_install(req):
    # Kaggle's own GPU builds of torch / torchaudio / numpy / scipy are kept.
    run(f"grep -vE '^(#|torch==|torchaudio==|numpy==|scipy==)' {req} > /tmp/req.txt && pip install -q -r /tmp/req.txt")


def find(pattern):
    """Newest match in the attached inputs (/kaggle/input) or in this run's output."""
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True) + glob.glob(f"{WORK}/{pattern}")
    hits.sort(key=os.path.getmtime)
    return hits[-1] if hits else None


def resume_arg(folder, marker):
    """' --resume-from <run folder>' when an attached earlier version has a checkpoint, else ''."""
    hits = glob.glob(f"/kaggle/input/**/{folder}/{marker}", recursive=True)
    if not hits:
        return ""
    newest = max(hits, key=os.path.getmtime)
    return " --resume-from " + newest[: newest.index(f"/{folder}/") + len(folder) + 1]


# ---- get the code (your repo) and the data/ folder (team repo)
shutil.rmtree(REPO, ignore_errors=True)
run(f"git clone --depth 1 -b {CODE_BRANCH} {CODE_REPO_URL} {REPO}")
if (REPO_URL, DATA_BRANCH) != (CODE_REPO_URL, CODE_BRANCH):
    fetched = subprocess.run(f"git fetch --depth 1 {REPO_URL} {DATA_BRANCH}", shell=True, cwd=REPO).returncode == 0
    has_data = fetched and subprocess.run("git ls-tree --name-only FETCH_HEAD data", shell=True, cwd=REPO,
                                          capture_output=True, text=True).stdout.strip()
    if has_data:
        run("git checkout FETCH_HEAD -- data")
    elif STEP == "smoke":
        print("The team repo has no data/ folder yet. That is fine for the smoke step.")
    else:
        raise SystemExit(f"No data/ folder in {REPO_URL} ({DATA_BRANCH}) yet. Only STEP = 'smoke' works without it.")
run("git log -1 --format='repo at %h %s' && nvidia-smi --query-gpu=name,memory.total --format=csv")
pip_install("xlsr_ctc/requirements.txt")
pip_install("baselines/scratch/requirements.txt")

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN secret found.")
except Exception:
    print("No HF_TOKEN secret: the Hugging Face upload is skipped.")
to_commit = []   # repo files that go into to_commit.zip

In [ ]:
XL, A2, A3 = WORK / "xlsr_run", WORK / "a2_ctc", WORK / "a3_rnnt"
TEST = "--split test --allow-test"


def upload_c2(model_dir):
    if HF_REPO and os.environ.get("HF_TOKEN"):
        run(f"python xlsr_ctc/push_to_hub.py --model-dir {model_dir} --repo {HF_REPO} --code-url {REPO_URL}")


def zip_folder(folder, name):
    if Path(folder).exists():
        shutil.make_archive(str(WORK / name), "zip", folder)
        print(f"Wrote {WORK / name}.zip (send it to Elif for C3)")


if STEP == "smoke":
    FAKE = WORK / "smoke"
    os.environ["ASR_OUTPUT_ROOT"] = str(FAKE / "outputs")   # fake results never go into benchmark/
    C = f"--clean-dir {FAKE}/data/clean"
    run(f"python common/make_fake_data.py --out {FAKE}/data --train 300 --dev 40 --min-words 6 --max-words 12 --long 8")
    run(f"python xlsr_ctc/train.py --out-dir {FAKE}/xlsr {C} --max-steps 40 --warmup-steps 10 --eval-every 20 --save-every 20")
    run(f"python xlsr_ctc/predict.py --model {FAKE}/xlsr/best {C} --split dev --logits-dir {FAKE}/logits")
    run(f"python baselines/scratch/train.py --model ctc --out-dir {FAKE}/a2 {C} --epochs 2 --warmup-steps 10")
    run(f"python baselines/scratch/train.py --model rnnt --out-dir {FAKE}/a3 {C} --epochs 2 --warmup-steps 10")
    run(f"python baselines/scratch/predict.py --ckpt {FAKE}/a3/best.pt {C} --split dev")
    shutil.rmtree(FAKE / "xlsr", ignore_errors=True)   # 4 GB of test checkpoints, not needed
    print("\nSMOKE TEST PASSED. The line '... s/step' above is the XLS-R speed on this GPU.")

elif STEP == "train_night":
    run_parallel([
        ("C2_xlsr", f"python xlsr_ctc/train.py --out-dir {XL} --max-hours {HOURS}" + resume_arg("xlsr_run", "last/trainer_state.json")),
        ("A2_ctc", f"python baselines/scratch/train.py --model ctc --out-dir {A2} --max-hours {HOURS}" + resume_arg("a2_ctc", "last.pt")),
    ])
    run(f"python xlsr_ctc/predict.py --model {XL}/best --split dev --logits-dir {WORK}/logits_dev")
    run(f"python baselines/scratch/predict.py --ckpt {A2}/best.pt --split dev")
    zip_folder(WORK / "logits_dev", "logits_dev_C2")
    shutil.copy(XL / "train_log.jsonl", REPO / "benchmark/results/C2_xlsr-300m-ctc_train_log.jsonl")
    upload_c2(XL / "best")
    to_commit += ["xlsr_ctc/requirements.txt", "baselines/scratch/requirements.txt"]

elif STEP == "train_day":
    run_parallel([
        ("A3_rnnt", f"python baselines/scratch/train.py --model rnnt --out-dir {A3} --max-hours {HOURS}" + resume_arg("a3_rnnt", "last.pt")),
    ])
    run(f"python baselines/scratch/predict.py --ckpt {A3}/best.pt --split dev")
    to_commit += ["baselines/scratch/requirements.txt"]

elif STEP == "test":
    xl_best = find("xlsr_run/best/best.json")
    a2_best, a3_best = find("a2_ctc/best.pt"), find("a3_rnnt/best.pt")
    if not (xl_best or HF_REPO) or not a2_best or not a3_best:
        raise SystemExit("Attach the outputs of the train_night and train_day versions first (Add Input).")
    c2_model = str(Path(xl_best).parent) if xl_best else HF_REPO
    run(f"python xlsr_ctc/predict.py --model {c2_model} {TEST} --logits-dir {WORK}/logits_test")
    run(f"python baselines/scratch/predict.py --ckpt {a2_best} {TEST}")
    run(f"python baselines/scratch/predict.py --ckpt {a3_best} {TEST}")
    zip_folder(WORK / "logits_test", "logits_test_C2")
    if xl_best:
        upload_c2(Path(xl_best).parent)

elif STEP == "upload":
    xl_best = find("xlsr_run/best/best.json")
    if not xl_best:
        raise SystemExit("Attach the output of the train_night version first (Add Input).")
    upload_c2(Path(xl_best).parent)

else:
    raise SystemExit(f"Unknown STEP {STEP!r}")

In [ ]:
# ---- to_commit.zip: extract it inside your local repo folder, then commit and push
if STEP != "smoke":
    for req in ["xlsr_ctc/requirements.txt", "baselines/scratch/requirements.txt"]:
        if req in to_commit:
            run(f"python common/pin_requirements.py {req}")   # exact versions that ran here
    for system in SYSTEMS:
        to_commit += [str(p.relative_to(REPO)) for p in (REPO / "benchmark").glob(f"*/{system}*")]
    with zipfile.ZipFile(WORK / "to_commit.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for rel in sorted(set(to_commit)):
            if (REPO / rel).exists():
                z.write(REPO / rel, rel)
                print("added", rel)
    print(f"\nDone. Download {WORK / 'to_commit.zip'} from the Output tab.")